In [1]:
from pathlib import Path
import sys
import time

ROOT = Path.cwd()
if not (ROOT / "python").exists() and (ROOT.parent / "python").exists():
    ROOT = ROOT.parent
EXAMPLE_DIR = ROOT / "example_lstm"

PYTHON_DIR = ROOT / "python"
BUILD_DIR = ROOT / "build"
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))
if BUILD_DIR.exists() and str(BUILD_DIR) not in sys.path:
    sys.path.insert(0, str(BUILD_DIR))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import backend_ndarray as nd  # noqa: E402
import nn  # noqa: E402
from optim import SGD  # noqa: E402
from example_lstm.data import Corpus, batchify, ensure_ptb  # noqa: E402
from example_lstm.training import epoch  # noqa: E402

Pick a backend. If the Metal extension is built and importable, `default_device()` will choose it; otherwise it falls back to NumPy.

In [2]:
device = nd.default_device()
device

metal()

Download Penn Treebank text files into `example_lstm/ptb` if they are missing, then build the dictionary and batchified arrays.

In [3]:
ptb_dir = ensure_ptb(EXAMPLE_DIR)
corpus = Corpus(ptb_dir)
batch_size = 16
seq_len = 40
train_data = batchify(corpus.train, batch_size=batch_size)
valid_data = batchify(corpus.valid, batch_size=batch_size)
{
    "vocab_size": len(corpus.dictionary),
    "train_shape": train_data.shape,
    "valid_shape": valid_data.shape,
}

{'vocab_size': 10000, 'train_shape': (58099, 16), 'valid_shape': (4610, 16)}

LSTM is the default here. Set `seq_model = "rnn"` to exercise the simpler RNN path with the same data and trainer.

In [4]:
seq_model = "lstm"
model = nn.LanguageModel(
    embedding_size=30,
    output_size=len(corpus.dictionary),
    hidden_size=64,
    num_layers=2,
    seq_model=seq_model,
    device=device,
)
len(model.parameters())

11

In [5]:
epochs = 2
opt = SGD(model.parameters(), lr=1.0)
history = []

for epoch_index in range(epochs):
    start = time.time()
    train_accuracy, train_loss = epoch(
        train_data,
        model,
        seq_len=seq_len,
        opt=opt,
        clip=0.25,
        device=device,
    )
    row = {
        "epoch": epoch_index + 1,
        "train_accuracy": train_accuracy,
        "train_loss": train_loss,
        "seconds": time.time() - start,
    }
    history.append(row)
    print(row)

history

{'epoch': 1, 'train_accuracy': 0.09350902784949568, 'train_loss': 6.550290722925612, 'seconds': 2483.6902780532837}
{'epoch': 2, 'train_accuracy': 0.13484758511480602, 'train_loss': 6.127344132668016, 'seconds': 2467.57635307312}


[{'epoch': 1,
  'train_accuracy': 0.09350902784949568,
  'train_loss': 6.550290722925612,
  'seconds': 2483.6902780532837},
 {'epoch': 2,
  'train_accuracy': 0.13484758511480602,
  'train_loss': 6.127344132668016,
  'seconds': 2467.57635307312}]

In [6]:
valid_accuracy, valid_loss = epoch(valid_data, model, seq_len=seq_len, device=device)
{"valid_accuracy": valid_accuracy, "valid_loss": valid_loss}

{'valid_accuracy': 0.1421810587980039, 'valid_loss': 6.028537774039442}